In [0]:
import os
import re
import pytesseract
import logging
import json
import pandas as pd
from PIL import Image

# -----------------------------------------------------------------------------
# Helper: Convert DBFS path to local path.
# -----------------------------------------------------------------------------
def dbfs_to_local_path(dbfs_path: str) -> str:
    if dbfs_path.startswith("dbfs:/"):
        return "/dbfs" + dbfs_path.replace("dbfs:", "")
    return dbfs_path

# -----------------------------------------------------------------------------
# PumpsPipeline Class: Encapsulates Pumps OCR and parsing logic.
# -----------------------------------------------------------------------------
class PumpsPipeline:
    @staticmethod
    def read_image(image_path: str) -> Image.Image:
        """
        Reads an image from a local or DBFS path and returns a PIL Image.
        """
        if image_path.startswith("dbfs:/"):
            local_path = "/dbfs" + image_path.replace("dbfs:", "")
        else:
            local_path = image_path
        if not os.path.exists(local_path):
            raise FileNotFoundError(f"File not found: {local_path}")
        img = Image.open(local_path)
        logging.info(f"Image loaded from {local_path} with size {img.size}")
        return img

    @staticmethod
    def perform_ocr(img: Image.Image) -> str:
        """
        Performs OCR on the given PIL image and returns the raw text.
        """
        text = pytesseract.image_to_string(img)
        logging.info("OCR extraction complete.")
        return text

    @staticmethod
    def parse_pumps_table(ocr_text: str) -> list:
        """
        Parses the pumps table from the OCR text.
        Expected format (each pump row):
          Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
          1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
          etc.
        """
        pump_pattern = re.compile(
            r"^(\d+)\s+(BOMCO)\s+(TRIPLEX)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
            re.IGNORECASE
        )
        pumps = []
        for line in ocr_text.splitlines():
            line = line.strip()
            match = pump_pattern.match(line)
            if match:
                number, model, pump_type, hhp, efficiency, stroke, liner, p_rating, p_limit, spm_rating, spm_limit = match.groups()
                pumps.append({
                    "Number": number,
                    "Model": model.upper(),
                    "Type": pump_type.upper(),
                    "HHP": hhp,
                    "Efficiency": efficiency,
                    "Stroke (in)": stroke,
                    "Liner (in)": liner,
                    "P-Rating (psi)": p_rating,
                    "P-Limit (psi)": p_limit,
                    "SPM Rating": spm_rating,
                    "SPM Limit": spm_limit
                })
        logging.info(f"Extracted {len(pumps)} pump rows.")
        return pumps

    @staticmethod
    def parse_drilling_circ_rates(ocr_text: str) -> list:
        """
        Parses drilling/circ rate lines from OCR text.
        Expected vertical tokens for each row are, for example:
          Drilling/Circ Rate 1
          4325 PS!
          @
          134
          SPM
          2.63 Gal/Stoke
          351.76 GPM
          8.38 BPM
          468.11 DC
          340.61 DP
          (then similarly for the next row)
        This function:
          - Finds the starting index of the drilling section (first token starting with "Drilling")
          - Groups subsequent tokens in blocks of 10 (if vertical format) and extracts numeric values.
        """
        tokens = [t.strip() for t in ocr_text.splitlines() if t.strip()]
        start_idx = None
        for i, token in enumerate(tokens):
            if token.lower().startswith("drilling") and "rate" in token.lower():
                start_idx = i
                break
        if start_idx is None:
            logging.warning("No drilling section found in OCR text.")
            return []
        drill_tokens = tokens[start_idx:]
        # Group tokens into blocks of 10 tokens.
        rows = []
        for i in range(0, len(drill_tokens), 10):
            group = drill_tokens[i:i+10]
            if len(group) < 10:
                break
            rows.append(group)
        parsed_rows = []
        for group in rows:
            # Extract the pump rate number from the first token.
            rate_match = re.search(r"(\d+)", group[0])
            rate_id = rate_match.group(1) if rate_match else ""
            # Token 1: pressure (extract numeric)
            pressure_match = re.search(r"([\d\.]+)", group[1])
            pressure = pressure_match.group(1) if pressure_match else ""
            # Token 3: SPM value (assuming a digit)
            spm = group[3] if group[3].isdigit() else ""
            # Token 5: Gal/Stoke (numeric part)
            gal_match = re.search(r"([\d\.]+)", group[5])
            gal_stroke = gal_match.group(1) if gal_match else ""
            # Token 6: GPM
            gpm_match = re.search(r"([\d\.]+)", group[6])
            gpm = gpm_match.group(1) if gpm_match else ""
            # Token 7: BPM
            bpm_match = re.search(r"([\d\.]+)", group[7])
            bpm = bpm_match.group(1) if bpm_match else ""
            # Token 8: DC
            dc_match = re.search(r"([\d\.]+)", group[8])
            dc = dc_match.group(1) if dc_match else ""
            # Token 9: DP
            dp_match = re.search(r"([\d\.]+)", group[9])
            dp = dp_match.group(1) if dp_match else ""
            parsed_rows.append({
                "RateID": rate_id,
                "Pressure": pressure,
                "SPM": spm,
                "Gal_Stroke": gal_stroke,
                "GPM": gpm,
                "BPM": bpm,
                "DC": dc,
                "DP": dp
            })
        logging.info(f"Extracted {len(parsed_rows)} drilling/circ rate rows.")
        return parsed_rows

    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        """
        Main processing method.
        Reads the image from the given path, performs OCR, then uses the above methods to parse
        the pumps table and the drilling/circ rates. Finally, builds a single dictionary with both sections.
        Returns a tuple (data_json, df) where:
            data_json = { "PUMPS": { "PUMPS": [pump rows], "DrillingCircRates": [drilling rows] } }
            df is a DataFrame of pump rows.
        """
        # Read image and perform OCR.
        img = PumpsPipeline.read_image(image_path)
        ocr_text = PumpsPipeline.perform_ocr(img)
        if debug:
            print("----- Full OCR Extracted Text -----")
            print(ocr_text)
        # Parse the pumps table.
        pumps = PumpsPipeline.parse_pumps_table(ocr_text)
        # Parse the drilling/circ rates.
        drilling = PumpsPipeline.parse_drilling_circ_rates(ocr_text)
        final_data = {
            "PUMPS": {
                "PUMPS": pumps,
                "DrillingCircRates": drilling
            }
        }
        # Convert pumps table to a DataFrame.
        df = pd.DataFrame(pumps)
        return final_data, df

# -----------------------------------------------------------------------------
# Main Integrated Pipeline for Pumps Section.
# -----------------------------------------------------------------------------
def main(debug: bool = False):
    # Example image path; adjust as needed.
    image_path = "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png"
    try:
        data_json, df = PumpsPipeline.process(image_path, debug=debug)
    except Exception as e:
        logger.error(e)
        return

    print("----- Pumps Extraction JSON Output -----")
    print(json.dumps(data_json, indent=4))
    # Save DataFrame and JSON.
    output_folder = dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
    os.makedirs(output_folder, exist_ok=True)
    pumps_csv = os.path.join(output_folder, "pumps.csv")
    json_path = os.path.join(output_folder, "pumps_drilling_circ.json")
    if not df.empty:
        df.to_csv(pumps_csv, index=False)
        logger.info(f"Pumps CSV saved to: {pumps_csv}")
    with open(json_path, "w") as f:
        json.dump(data_json, f, indent=4)
    logger.info(f"JSON saved to: {json_path}")
    print("----- Aggregated JSON Output -----")
    print(json.dumps(data_json, indent=4))

# -----------------------------------------------------------------------------
# Entry Point
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")
    main(debug=True)


INFO: Pumps CSV saved to: /dbfs/mnt/mini-proj-dd/final_results/pumps.csv
INFO:PumpExtractor:Pumps CSV saved to: /dbfs/mnt/mini-proj-dd/final_results/pumps.csv
INFO: JSON saved to: /dbfs/mnt/mini-proj-dd/final_results/pumps_drilling_circ.json
INFO:PumpExtractor:JSON saved to: /dbfs/mnt/mini-proj-dd/final_results/pumps_drilling_circ.json


----- Full OCR Extracted Text -----
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

----- Pumps Extraction JSON Output -----
{
    "PUMPS": {
        "PUMPS": [
            {
                "Number": "1",
                "Model": "BOMCO",
                "Type": "TRIPLEX",
                "HHP": "1600",
                "Efficiency": "95",
                "Stroke (in)": "12.000",
                "Liner (in)": "4.75",
                "P-Rating (psi)": "7500",
                "P-Limit (psi)": "7100",
                "SPM Rating": "120

In [0]:
import os
import re
import pytesseract
import logging
import json
import pandas as pd
from PIL import Image

# -----------------------------------------------------------------------------
# Helper: Convert DBFS path to local path.
# -----------------------------------------------------------------------------
def dbfs_to_local_path(dbfs_path: str) -> str:
    if dbfs_path.startswith("dbfs:/"):
        return "/dbfs" + dbfs_path.replace("dbfs:", "")
    return dbfs_path

# -----------------------------------------------------------------------------
# Read Image from a local or DBFS path.
# -----------------------------------------------------------------------------
def read_image(image_path):
    """
    Reads the image from local or DBFS path and returns a PIL Image.
    """
    if image_path.startswith("dbfs:/"):
        local_path = "/dbfs" + image_path.replace("dbfs:", "")
    else:
        local_path = image_path

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    
    img = Image.open(local_path)
    logger.info(f"Image loaded from {local_path} with size {img.size}")
    return img

# -----------------------------------------------------------------------------
# Perform OCR on a PIL Image and return the full text.
# -----------------------------------------------------------------------------
def perform_ocr(img):
    text = pytesseract.image_to_string(img)
    logger.info("OCR extraction complete.")
    return text

# -----------------------------------------------------------------------------
# Parse Pumps Table from OCR text using regex.
# -----------------------------------------------------------------------------
def parse_pumps_table(ocr_text):
    """
    Parses the pumps table from the OCR text.
    Expected format (example):
      Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
      1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
      2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
      3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
    """
    pump_pattern = re.compile(
        r"^(\d+)\s+(BOMCO)\s+(TRIPLEX)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
        re.IGNORECASE
    )
    pumps = []
    for line in ocr_text.splitlines():
        line = line.strip()
        match = pump_pattern.match(line)
        if match:
            number, model, pump_type, hhp, efficiency, stroke, liner, p_rating, p_limit, spm_rating, spm_limit = match.groups()
            pumps.append({
                "Number": number,
                "Model": model.upper(),
                "Type": pump_type.upper(),
                "HHP": hhp,
                "Efficiency": efficiency,
                "Stroke (in)": stroke,
                "Liner (in)": liner,
                "P-Rating (psi)": p_rating,
                "P-Limit (psi)": p_limit,
                "SPM Rating": spm_rating,
                "SPM Limit": spm_limit
            })
    logger.info(f"Extracted {len(pumps)} pump rows.")
    return pumps

# -----------------------------------------------------------------------------
# Parse Drilling/Circ Rates from OCR text in vertical token format.
# -----------------------------------------------------------------------------
def parse_drilling_circ_rates_vertical(ocr_text):
    """
    Parses drilling/circ rate data from OCR text in vertical format.
    Expected vertical tokens (one token per line) for each valid drilling row:
      Drilling/Circ Rate 1
      4325 PS!
      @
      134
      SPM
      2.63 Gal/Stoke
      351.76 GPM
      8.38 BPM
      468.11 DC
      340.61 DP
      ... (then similarly for the next row)
    This function searches for the start of the drilling section and groups subsequent tokens in blocks of 10.
    Then it extracts the fields.
    """
    tokens = [t.strip() for t in ocr_text.splitlines() if t.strip()]
    # Find the start token for drilling (look for first token starting with "drilling" and containing "rate")
    start_idx = None
    for i, token in enumerate(tokens):
        if token.lower().startswith("drilling") and "rate" in token.lower():
            start_idx = i
            break
    if start_idx is None:
        logger.warning("No drilling section found in OCR text.")
        return []
    
    drill_tokens = tokens[start_idx:]
    # Group tokens into blocks of 10 (each valid row should have 10 tokens)
    rows = []
    for i in range(0, len(drill_tokens), 10):
        group = drill_tokens[i:i+10]
        if len(group) < 10:
            continue
        rows.append(group)
    
    parsed_rows = []
    for group in rows:
        # Extract the pump rate from the first token
        rate_match = re.search(r"(\d+)", group[0])
        rate_id = rate_match.group(1) if rate_match else ""
        # Token 2 may be a filler (@); we discard it.
        # Assume tokens:
        # [0]: "Drilling/Circ Rate X"  -> X as RateID
        # [1]: Pressure with PSI (e.g. "4325 PS!")
        # [3]: SPM (e.g. "134")
        # [5]: Gal/Stoke (e.g. "2.63 Gal/Stoke") -> extract numeric part
        # [6]: GPM (e.g. "351.76 GPM") -> numeric
        # [7]: BPM (e.g. "8.38 BPM") -> numeric
        # [8]: DC (e.g. "468.11 DC") -> numeric
        # [9]: DP (e.g. "340.61 DP") -> numeric
        pressure_match = re.search(r"([\d\.]+)", group[1])
        pressure = pressure_match.group(1) if pressure_match else ""
        spm = group[3] if group[3].isdigit() else ""
        gal_stroke_match = re.search(r"([\d\.]+)", group[5])
        gal_stroke = gal_stroke_match.group(1) if gal_stroke_match else ""
        gpm_match = re.search(r"([\d\.]+)", group[6])
        gpm = gpm_match.group(1) if gpm_match else ""
        bpm_match = re.search(r"([\d\.]+)", group[7])
        bpm = bpm_match.group(1) if bpm_match else ""
        dc_match = re.search(r"([\d\.]+)", group[8])
        dc = dc_match.group(1) if dc_match else ""
        dp_match = re.search(r"([\d\.]+)", group[9])
        dp = dp_match.group(1) if dp_match else ""
        parsed_rows.append({
            "RateID": rate_id,
            "Pressure": pressure,
            "SPM": spm,
            "Gal_Stroke": gal_stroke,
            "GPM": gpm,
            "BPM": bpm,
            "DC": dc,
            "DP": dp,
            "Raw": " ".join(group)
        })
    logger.info(f"Extracted {len(parsed_rows)} drilling/circ rate rows.")
    return parsed_rows

# -----------------------------------------------------------------------------
# Main Pipeline Function
# -----------------------------------------------------------------------------
def main_pipeline():
    # Set the image path (adjust as needed)
    image_path = "/dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png"
    try:
        img = read_image(image_path)
    except FileNotFoundError as e:
        logger.error(e)
        return

    # 1) Perform OCR on the entire image.
    ocr_text = perform_ocr(img)
    logger.info(f"OCR Text:\n{ocr_text}\n")
    print("----- Full OCR Extracted Text -----")
    print(ocr_text)

    # 2) Parse the pumps table.
    pumps = parse_pumps_table(ocr_text)
    logger.info(f"Extracted Pumps Table: {pumps}")
    print("----- Extracted Pumps Table -----")
    print(json.dumps(pumps, indent=4))

    # 3) Parse the drilling/circ rates using vertical tokens.
    circ_rates = parse_drilling_circ_rates_vertical(ocr_text)
    logger.info(f"Extracted Drilling/Circ Rates: {circ_rates}")
    print("----- Extracted Drilling/Circ Rates -----")
    print(json.dumps(circ_rates, indent=4))

    # 4) Build final JSON structure with both sections nested under "PUMPS"
    final_data = {
        "PUMPS": {
            "PUMPS": pumps,
            "DrillingCircRates": circ_rates
        }
    }
    logger.info("=== Final JSON ===")
    final_json = json.dumps(final_data, indent=4)
    print(final_json)

    # 5) Save results
    output_folder = "/dbfs/mnt/mini-proj-dd/final_results"
    os.makedirs(output_folder, exist_ok=True)
    pumps_csv = os.path.join(output_folder, "pumps.csv")
    circ_csv = os.path.join(output_folder, "drilling_circ_rates.csv")
    json_path = os.path.join(output_folder, "pumps_drilling_circ.json")
    
    if pumps:
        import pandas as pd
        df_pumps = pd.DataFrame(pumps)
        df_pumps.to_csv(pumps_csv, index=False)
        logger.info(f"Pumps CSV saved to: {pumps_csv}")
    if circ_rates:
        import pandas as pd
        df_circ = pd.DataFrame(circ_rates)
        df_circ.to_csv(circ_csv, index=False)
        logger.info(f"Drilling/Circ Rates CSV saved to: {circ_csv}")
    with open(json_path, "w") as f:
        json.dump(final_data, f, indent=4)
    logger.info(f"JSON saved to: {json_path}")

# -----------------------------------------------------------------------------
# Minimal Logger Configuration (if not set elsewhere)
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")
    main_pipeline()


INFO: Image loaded from /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png with size (2502, 276)
INFO:PumpExtractor:Image loaded from /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png with size (2502, 276)
INFO: OCR extraction complete.
INFO:PumpExtractor:OCR extraction complete.
INFO: OCR Text:
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP


INFO:PumpExtractor:OCR Text:
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 710

----- Full OCR Extracted Text -----
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

----- Extracted Pumps Table -----
[
    {
        "Number": "1",
        "Model": "BOMCO",
        "Type": "TRIPLEX",
        "HHP": "1600",
        "Efficiency": "95",
        "Stroke (in)": "12.000",
        "Liner (in)": "4.75",
        "P-Rating (psi)": "7500",
        "P-Limit (psi)": "7100",
        "SPM Rating": "120",
        "SPM Limit": "110"
    },
    {
        "Number": "2",
        "Model": "BOMCO",
        "Type": "TRIPLEX",
        "H

In [0]:
import os
import re
import pytesseract
import logging
import json
import pandas as pd
from PIL import Image

# -----------------------------------------------------------------------------
# Helper: Convert DBFS path to local path.
# -----------------------------------------------------------------------------
def dbfs_to_local_path(dbfs_path: str) -> str:
    if dbfs_path.startswith("dbfs:/"):
        return "/dbfs" + dbfs_path.replace("dbfs:", "")
    return dbfs_path

# -----------------------------------------------------------------------------
# Read Image from a local or DBFS path.
# -----------------------------------------------------------------------------
def read_image(image_path):
    """
    Reads the image from local or DBFS path and returns a PIL Image.
    """
    if image_path.startswith("dbfs:/"):
        local_path = "/dbfs" + image_path.replace("dbfs:", "")
    else:
        local_path = image_path

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    
    img = Image.open(local_path)
    logger.info(f"Image loaded from {local_path} with size {img.size}")
    return img

# -----------------------------------------------------------------------------
# Perform OCR on a PIL Image and return the full text.
# -----------------------------------------------------------------------------
def perform_ocr(img):
    text = pytesseract.image_to_string(img)
    logger.info("OCR extraction complete.")
    return text

# -----------------------------------------------------------------------------
# Parse Pumps Table from OCR text using regex.
# -----------------------------------------------------------------------------
def parse_pumps_table(ocr_text):
    """
    Parses the pumps table from the OCR text.
    Expected format (example):
      Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
      1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
      2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
      3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
    """
    pump_pattern = re.compile(
        r"^(\d+)\s+(BOMCO)\s+(TRIPLEX)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
        re.IGNORECASE
    )
    pumps = []
    for line in ocr_text.splitlines():
        line = line.strip()
        match = pump_pattern.match(line)
        if match:
            number, model, pump_type, hhp, efficiency, stroke, liner, p_rating, p_limit, spm_rating, spm_limit = match.groups()
            pumps.append({
                "Number": number,
                "Model": model.upper(),
                "Type": pump_type.upper(),
                "HHP": hhp,
                "Efficiency": efficiency,
                "Stroke (in)": stroke,
                "Liner (in)": liner,
                "P-Rating (psi)": p_rating,
                "P-Limit (psi)": p_limit,
                "SPM Rating": spm_rating,
                "SPM Limit": spm_limit
            })
    logger.info(f"Extracted {len(pumps)} pump rows.")
    return pumps

# -----------------------------------------------------------------------------
# Parse Drilling/Circ Rates from OCR text in vertical token format.
# -----------------------------------------------------------------------------
def parse_drilling_circ_rates_vertical(ocr_text):
    """
    Parses drilling/circ rate data from OCR text in vertical format.
    Expected vertical tokens (one token per line) for each valid drilling row:
      Drilling/Circ Rate 1
      4325 PS!
      @
      134
      SPM
      2.63 Gal/Stoke
      351.76 GPM
      8.38 BPM
      468.11 DC
      340.61 DP
      ... (then similarly for the next row)
    This function searches for the start of the drilling section and groups subsequent tokens in blocks of 10.
    Then it extracts the fields.
    """
    tokens = [t.strip() for t in ocr_text.splitlines() if t.strip()]
    # Find the start token for drilling (look for first token starting with "drilling" and containing "rate")
    start_idx = None
    for i, token in enumerate(tokens):
        if token.lower().startswith("drilling") and "rate" in token.lower():
            start_idx = i
            break
    if start_idx is None:
        logger.warning("No drilling section found in OCR text.")
        return []
    
    drill_tokens = tokens[start_idx:]
    # Group tokens into blocks of 10 (each valid row should have 10 tokens)
    rows = []
    for i in range(0, len(drill_tokens), 10):
        group = drill_tokens[i:i+10]
        if len(group) < 10:
            continue
        rows.append(group)
    
    parsed_rows = []
    for group in rows:
        # Extract the pump rate from the first token
        rate_match = re.search(r"(\d+)", group[0])
        rate_id = rate_match.group(1) if rate_match else ""
        # Token 2 may be a filler (@); we discard it.
        # Assume tokens:
        # [0]: "Drilling/Circ Rate X"  -> X as RateID
        # [1]: Pressure with PSI (e.g. "4325 PS!")
        # [3]: SPM (e.g. "134")
        # [5]: Gal/Stoke (e.g. "2.63 Gal/Stoke") -> extract numeric part
        # [6]: GPM (e.g. "351.76 GPM") -> numeric
        # [7]: BPM (e.g. "8.38 BPM") -> numeric
        # [8]: DC (e.g. "468.11 DC") -> numeric
        # [9]: DP (e.g. "340.61 DP") -> numeric
        pressure_match = re.search(r"([\d\.]+)", group[1])
        pressure = pressure_match.group(1) if pressure_match else ""
        spm = group[3] if group[3].isdigit() else ""
        gal_stroke_match = re.search(r"([\d\.]+)", group[5])
        gal_stroke = gal_stroke_match.group(1) if gal_stroke_match else ""
        gpm_match = re.search(r"([\d\.]+)", group[6])
        gpm = gpm_match.group(1) if gpm_match else ""
        bpm_match = re.search(r"([\d\.]+)", group[7])
        bpm = bpm_match.group(1) if bpm_match else ""
        dc_match = re.search(r"([\d\.]+)", group[8])
        dc = dc_match.group(1) if dc_match else ""
        dp_match = re.search(r"([\d\.]+)", group[9])
        dp = dp_match.group(1) if dp_match else ""
        parsed_rows.append({
            "RateID": rate_id,
            "Pressure": pressure,
            "SPM": spm,
            "Gal_Stroke": gal_stroke,
            "GPM": gpm,
            "BPM": bpm,
            "DC": dc,
            "DP": dp,
            "Raw": " ".join(group)
        })
    logger.info(f"Extracted {len(parsed_rows)} drilling/circ rate rows.")
    return parsed_rows

# -----------------------------------------------------------------------------
# Main Pipeline Function
# -----------------------------------------------------------------------------
def main_pipeline():
    # Set the image path (adjust as needed)
    image_path = "/dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png"
    try:
        img = read_image(image_path)
    except FileNotFoundError as e:
        logger.error(e)
        return

    # 1) Perform OCR on the entire image.
    ocr_text = perform_ocr(img)
    logger.info(f"OCR Text:\n{ocr_text}\n")
    print("----- Full OCR Extracted Text -----")
    print(ocr_text)

    # 2) Parse the pumps table.
    pumps = parse_pumps_table(ocr_text)
    logger.info(f"Extracted Pumps Table: {pumps}")
    print("----- Extracted Pumps Table -----")
    print(json.dumps(pumps, indent=4))

    # 3) Parse the drilling/circ rates using vertical tokens.
    circ_rates = parse_drilling_circ_rates_vertical(ocr_text)
    logger.info(f"Extracted Drilling/Circ Rates: {circ_rates}")
    print("----- Extracted Drilling/Circ Rates -----")
    print(json.dumps(circ_rates, indent=4))

    # 4) Build final JSON structure with both sections nested under "PUMPS"
    final_data = {
        "PUMPS": {
            "PUMPS": pumps,
            "DrillingCircRates": circ_rates
        }
    }
    logger.info("=== Final JSON ===")
    final_json = json.dumps(final_data, indent=4)
    print(final_json)

    # 5) Save results
    output_folder = "/dbfs/mnt/mini-proj-dd/final_results"
    os.makedirs(output_folder, exist_ok=True)
    pumps_csv = os.path.join(output_folder, "pumps.csv")
    circ_csv = os.path.join(output_folder, "drilling_circ_rates.csv")
    json_path = os.path.join(output_folder, "pumps_drilling_circ.json")
    
    if pumps:
        import pandas as pd
        df_pumps = pd.DataFrame(pumps)
        df_pumps.to_csv(pumps_csv, index=False)
        logger.info(f"Pumps CSV saved to: {pumps_csv}")
    if circ_rates:
        import pandas as pd
        df_circ = pd.DataFrame(circ_rates)
        df_circ.to_csv(circ_csv, index=False)
        logger.info(f"Drilling/Circ Rates CSV saved to: {circ_csv}")
    with open(json_path, "w") as f:
        json.dump(final_data, f, indent=4)
    logger.info(f"JSON saved to: {json_path}")

# -----------------------------------------------------------------------------
# Minimal Logger Configuration (if not set elsewhere)
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")
    main_pipeline()


INFO: Image loaded from /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png with size (2502, 276)
INFO:PumpExtractor:Image loaded from /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png with size (2502, 276)
INFO: OCR extraction complete.
INFO:PumpExtractor:OCR extraction complete.
INFO: OCR Text:
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP


INFO:PumpExtractor:OCR Text:
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 710

----- Full OCR Extracted Text -----
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

----- Extracted Pumps Table -----
[
    {
        "Number": "1",
        "Model": "BOMCO",
        "Type": "TRIPLEX",
        "HHP": "1600",
        "Efficiency": "95",
        "Stroke (in)": "12.000",
        "Liner (in)": "4.75",
        "P-Rating (psi)": "7500",
        "P-Limit (psi)": "7100",
        "SPM Rating": "120",
        "SPM Limit": "110"
    },
    {
        "Number": "2",
        "Model": "BOMCO",
        "Type": "TRIPLEX",
        "H

In [0]:
import os
import re
import cv2
import json
import logging
import numpy as np
import pytesseract
import pandas as pd
from PIL import Image

# -----------------------------------------------------------------------------
# Helper: Convert DBFS path to a local path.
# -----------------------------------------------------------------------------
def dbfs_to_local_path(dbfs_path: str) -> str:
    if dbfs_path.startswith("dbfs:/"):
        return "/dbfs" + dbfs_path.replace("dbfs:", "")
    return dbfs_path

# -----------------------------------------------------------------------------
# PumpsPipeline Class: Encapsulates all Pumps OCR and parsing logic.
# -----------------------------------------------------------------------------
class PumpsPipeline:
    # Regex for drilling/circ-related keywords.
    DRILLING_REGEX = re.compile(r"(drilling|circ|annular)", re.IGNORECASE)

    @staticmethod
    def enhance_image(img: np.ndarray) -> np.ndarray:
        """Enhance image (grayscale, histogram equalization, and bilateral filter)."""
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        eq = cv2.equalizeHist(gray)
        filtered = cv2.bilateralFilter(eq, d=9, sigmaColor=75, sigmaSpace=75)
        return filtered

    @staticmethod
    def binarize_image(img: np.ndarray) -> (np.ndarray, np.ndarray):
        """Return both Otsu and adaptive thresholded images."""
        _, otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        adaptive = cv2.adaptiveThreshold(img, 255,
                                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY, 15, 10)
        return otsu, adaptive

    @staticmethod
    def detect_rows_via_morph_ops(img: np.ndarray, debug: bool = False) -> list:
        """Detect row boundaries using morphological operations."""
        logger = logging.getLogger("PumpsPipeline.detect_rows_via_morph_ops")
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        morph = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=2)
        inv = cv2.bitwise_not(morph)
        contours, _ = cv2.findContours(inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 50 and h < 20:
                boxes.append((x, y, w, h))
        boxes.sort(key=lambda b: b[1])
        y_coords = []
        for (x, y, w, h) in boxes:
            y_coords.extend([y, y+h])
        y_coords = sorted(list(set(y_coords)))
        h_img, w_img = gray.shape
        if y_coords and y_coords[0] > 5:
            y_coords.insert(0, 0)
        if y_coords and abs(y_coords[-1]-h_img) > 5:
            y_coords.append(h_img)
        row_boxes = []
        for i in range(len(y_coords)-1):
            if y_coords[i+1]-y_coords[i] >= 10:
                row_boxes.append((0, y_coords[i], w_img, y_coords[i+1]-y_coords[i]))
        if debug:
            logger.debug(f"Detected row boxes: {row_boxes}")
        return row_boxes

    @staticmethod
    def ocr_on_rows(img: np.ndarray, row_boxes: list, debug: bool = False) -> list:
        """Perform OCR (--psm 6) on each detected row and return the resulting texts."""
        logger = logging.getLogger("PumpsPipeline.ocr_on_rows")
        texts = []
        for i, (x, y, w, h) in enumerate(row_boxes):
            roi = img[y:y+h, x:x+w]
            gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            pil_roi = Image.fromarray(bw)
            text = pytesseract.image_to_string(pil_roi, config="--psm 6").strip()
            texts.append(text)
            if debug:
                logger.debug(f"OCR Row {i+1}: {text}")
                print(f"OCR Row {i+1}: {text}")
        return texts

    @staticmethod
    def multi_psm_ocr(roi: np.ndarray, debug: bool = False) -> str:
        """Run OCR on ROI using both PSM 6 and 11, and return the best result."""
        logger = logging.getLogger("PumpsPipeline.multi_psm_ocr")
        enhanced = PumpsPipeline.enhance_image(roi)
        otsu, _ = PumpsPipeline.binarize_image(enhanced)
        pil_img = Image.fromarray(otsu)
        psm_modes = ["6", "11"]
        results = {}
        for mode in psm_modes:
            config = f"--psm {mode}"
            text = pytesseract.image_to_string(pil_img, config=config).strip()
            lines = [l for l in text.splitlines() if l.strip()]
            results[mode] = (text, lines)
            if debug:
                logger.debug(f"[PSM {mode}] extracted {len(lines)} lines: {lines}")
                print(f"[PSM {mode}] extracted lines: {lines}")
        best_mode = max(psm_modes, key=lambda m: sum(1 for l in results[m][1] if l.strip() and l.strip()[0].isdigit()))
        logger.debug(f"Selected best PSM mode: {best_mode}")
        return results[best_mode][0]

    @staticmethod
    def segment_rows_via_projection(roi: np.ndarray, debug: bool = False) -> list:
        """Segment the ROI into rows using horizontal projection."""
        logger = logging.getLogger("PumpsPipeline.segment_rows_via_projection")
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        proj = np.sum(binary, axis=1)
        gap_thresh = np.max(proj) * 0.2
        gap_indices = np.where(proj < gap_thresh)[0]
        if len(gap_indices) == 0:
            pil_img = Image.fromarray(binary)
            row_text = pytesseract.image_to_string(pil_img, config="--psm 7").strip()
            logger.debug("No gaps detected; using full ROI as one row.")
            return [row_text]
        gap_segments = []
        start = gap_indices[0]
        prev = gap_indices[0]
        for idx in gap_indices[1:]:
            if idx - prev > 1:
                gap_segments.append((start, prev))
                start = idx
            prev = idx
        gap_segments.append((start, prev))
        boundaries = [int((s+e)//2) for s, e in gap_segments]
        boundaries = [0] + boundaries + [binary.shape[0]]
        row_texts = []
        for i in range(len(boundaries)-1):
            r1, r2 = boundaries[i], boundaries[i+1]
            if r2 - r1 < 10:
                continue
            row_img = roi[r1:r2, :]
            pil_row = Image.fromarray(row_img)
            text = pytesseract.image_to_string(pil_row, config="--psm 7").strip()
            if text:
                row_texts.append(text)
        logger.debug(f"Projection segmentation produced {len(row_texts)} rows: {row_texts}")
        print("Projection segmentation output:", row_texts)
        return row_texts

    @staticmethod
    def split_block_to_sections(block_text: str, debug: bool = False) -> (list, list):
        """
        Split combined OCR block text into pump and drilling sections.
        Any line that contains drilling-related keywords is sent solely to drilling.
        """
        logger = logging.getLogger("PumpsPipeline.split_block_to_sections")
        lines = [l.strip() for l in block_text.splitlines() if l.strip()]
        pump_lines = []
        drilling_lines = []
        for line in lines:
            if PumpsPipeline.DRILLING_REGEX.search(line):
                drilling_lines.append(line)
            else:
                pump_lines.append(line)
        if debug:
            logger.debug(f"Split block: Pump lines: {pump_lines}")
            logger.debug(f"Split block: Drilling lines: {drilling_lines}")
            print("Pump lines from block:", pump_lines)
            print("Drilling lines from block:", drilling_lines)
        return pump_lines, drilling_lines

    @staticmethod
    def parse_pump_lines(lines: list, debug: bool = False) -> list:
        """
        Parse pump lines into structured pump rows.
        Look for a header row (containing "Number" and "Model").
        Then only process subsequent lines that start with a digit.
        Log token-by-token extraction.
        """
        logger = logging.getLogger("PumpsPipeline.parse_pump_lines")
        # Exclude any line that contains drilling keywords.
        filtered = [l for l in lines if not PumpsPipeline.DRILLING_REGEX.search(l)]
        logger.debug(f"Filtered pump lines: {filtered}")
        header_idx = None
        for idx, line in enumerate(filtered):
            low = line.lower()
            if "number" in low and "model" in low:
                header_idx = idx
                logger.debug(f"Found pump header at index {idx}: {line}")
                break
        if header_idx is None:
            logger.error("Pump header not detected in OCR text.")
            print("Pump header not detected in OCR text!")
            return []
        pump_candidates = filtered[header_idx+1:]
        logger.debug(f"Pump candidate lines: {pump_candidates}")
        pump_rows = []
        for line in pump_candidates:
            if not line[0].isdigit():
                logger.debug(f"Skipping non-pump line: {line}")
                continue
            tokens = re.split(r'\s{2,}|\|', line)
            tokens = [t.strip() for t in tokens if t.strip()]
            logger.debug(f"Tokens from line: {tokens}")
            if len(tokens) < 11:
                logger.warning(f"Skipping line (insufficient tokens): {line}")
                continue
            row = {
                "Number": tokens[0],
                "Model": tokens[1],
                "Type": tokens[2],
                "HHP": tokens[3],
                "Efficiency": tokens[4],
                "Stroke (in)": tokens[5],
                "Liner (in)": tokens[6],
                "P-Rating (psi)": tokens[7],
                "P-Limit (psi)": tokens[8],
                "SPM Rating": tokens[9],
                "SPM Limit": tokens[10]
            }
            pump_rows.append(row)
            logger.debug(f"Parsed pump row: {row}")
        logger.debug(f"Final parsed pump rows: {pump_rows}")
        print("Final parsed pump rows:", pump_rows)
        return pump_rows

    @staticmethod
    def parse_drilling_lines(lines: list, debug: bool = False) -> list:
        """
        Parse drilling/circ rate lines into structured rows.
        Extract exactly eight numeric values (RateID, Pressure, SPM, Gal_Stroke, GPM, BPM, DC, DP).
        """
        logger = logging.getLogger("PumpsPipeline.parse_drilling_lines")
        drilling_rows = []
        for line in lines:
            numbers = re.findall(r"[\d\.]+", line)
            if len(numbers) < 8:
                logger.warning(f"Drilling line skipped (insufficient numbers): {line}")
                continue
            row = {
                "RateID": numbers[0],
                "Pressure": numbers[1],
                "SPM": numbers[2],
                "Gal_Stroke": numbers[3],
                "GPM": numbers[4],
                "BPM": numbers[5],
                "DC": numbers[6],
                "DP": numbers[7],
                "Raw": line
            }
            drilling_rows.append(row)
            logger.debug(f"Parsed drilling row: {row}")
        logger.debug(f"Final parsed drilling rows: {drilling_rows}")
        return drilling_rows

    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        """
        Main processing method.
        Reads an image from the given DBFS path, applies OCR row by row, splits into pump and drilling sections,
        and returns a tuple (data_json, df) where data_json = { "PUMPS": [pump rows], "DrillingCircRates": [drilling rows] }
        and df is a DataFrame of pump rows.
        """
        logger = logging.getLogger("PumpsPipeline.process")
        local_path = dbfs_to_local_path(image_path)
        img = cv2.imread(local_path)
        if img is None:
            raise FileNotFoundError(f"Unable to read image at: {image_path}")
        logger.info(f"Image loaded from {local_path} with size {img.shape[:2]}.")

        row_boxes = PumpsPipeline.detect_rows_via_morph_ops(img, debug=debug)
        logger.info(f"Detected {len(row_boxes)} row boxes.")
        row_texts = PumpsPipeline.ocr_on_rows(img, row_boxes, debug=debug)
        logger.info("OCR results for each row:")
        for i, text in enumerate(row_texts):
            logger.info(f"Row {i+1}: {text}")
        block_text = "\n".join(row_texts)
        logger.debug(f"Combined OCR block text:\n{block_text}")

        pump_lines, drilling_lines = PumpsPipeline.split_block_to_sections(block_text, debug=debug)
        logger.info(f"Block split: {len(pump_lines)} pump lines, {len(drilling_lines)} drilling lines.")

        # Optional: Print out the OCR extracted lines
        print("Full OCR Pump Lines:")
        for line in pump_lines:
            print(line)
        print("Full OCR Drilling Lines:")
        for line in drilling_lines:
            print(line)

        # Attempt re-segmentation using pump ROI (by looking for a header line with "Number" and "Model").
        pump_roi = None
        for i, text in enumerate(row_texts):
            if "number" in text.lower() and "model" in text.lower():
                box = row_boxes[i]
                pump_roi = img[box[1]:box[1]+box[3], box[0]:box[0]+box[2]]
                logger.info("Pump ROI found for re-segmentation.")
                break
        if pump_roi is not None:
            pump_block = PumpsPipeline.multi_psm_ocr(pump_roi, debug=debug)
            resegmented = PumpsPipeline.segment_rows_via_projection(pump_roi, debug=debug)
            final_pump_lines = resegmented if len(resegmented) > len(pump_block.splitlines()) else pump_block.splitlines()
            pump_lines = final_pump_lines
            logger.info(f"Re-segmentation yielded {len(pump_lines)} pump lines.")
            print("Final pump lines after re-segmentation:")
            for line in pump_lines:
                print(line)
        else:
            logger.info("No dedicated pump ROI found; using initial pump lines.")

        pumps = PumpsPipeline.parse_pump_lines(pump_lines, debug=debug)
        drilling = PumpsPipeline.parse_drilling_lines(drilling_lines, debug=debug)
        logger.info(f"Parsed {len(pumps)} pump rows and {len(drilling)} drilling rows.")
        data_json = {"PUMPS": pumps, "DrillingCircRates": drilling}
        df = pd.DataFrame(pumps)
        return data_json, df

# -----------------------------------------------------------------------------
# PipelineRunner Class: Integrates PumpsPipeline into a full pipeline.
# -----------------------------------------------------------------------------
class PipelineRunner:
    def __init__(self, image_path: str, debug: bool = False):
        self.image_path = image_path
        self.debug = debug
        self.logger = logging.getLogger("PipelineRunner")
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
            self.logger.addHandler(handler)
    
    def run(self):
        try:
            data_json, df = PumpsPipeline.process(self.image_path, debug=self.debug)
            print("----- Pumps Extraction JSON Output -----")
            print(json.dumps(data_json, indent=4))
            df.to_csv("pumps_extracted.csv", index=False)
            self.logger.info("pumps_extracted.csv saved.")
            df_drilling = pd.DataFrame(data_json["DrillingCircRates"])
            df_drilling.to_csv("drilling_rates_extracted.csv", index=False)
            self.logger.info("drilling_rates_extracted.csv saved.")
        except Exception as e:
            self.logger.exception("Error during pipeline processing.")

# -----------------------------------------------------------------------------
# Final Integrated Main Pipeline
# -----------------------------------------------------------------------------
def main(debug: bool = False):
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    logger.info("Starting main pipeline execution for Pumps section...")

    image_paths = {
        "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png"
    }

    pipelines = {
        "PUMPS": PumpsPipeline.process
    }

    output_folder = dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
    os.makedirs(output_folder, exist_ok=True)
    aggregated_json = {}
    aggregated_df = pd.DataFrame()

    for section, path in image_paths.items():
        func = pipelines.get(section)
        if not func:
            logger.info(f"Skipping section '{section}' — pipeline not implemented.")
            continue
        try:
            logger.info(f"Processing section '{section}' from {path}...")
            data_json, df = func(path, debug=debug)
            safe_section = section.replace(" ", "_").lower()
            json_file = os.path.join(output_folder, f"{safe_section}.json")
            with open(json_file, "w") as f:
                json.dump(data_json, f, indent=4)
            if df is not None and not df.empty:
                aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
                csv_file = os.path.join(output_folder, f"{safe_section}.csv")
                df.to_csv(csv_file, index=False)
            aggregated_json[section] = data_json
            logger.info(f"Section '{section}' processed successfully.")
        except Exception as e:
            logger.exception(f"Error processing section '{section}': {e}")

    agg_json_path = os.path.join(output_folder, "aggregated_data.json")
    with open(agg_json_path, "w") as f:
        json.dump(aggregated_json, f, indent=4)
    agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
    aggregated_df.to_csv(agg_csv_path, index=False)
    logger.info(f"Aggregated results saved to {agg_json_path} and {agg_csv_path}.")
    print("----- Aggregated JSON Output -----")
    print(json.dumps(aggregated_json, indent=4))

# -----------------------------------------------------------------------------
# Entry Point
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    # For detailed logging (including OCR extracted text), run with debug=True
    main(debug=True)


INFO:__main__:Starting main pipeline execution for Pumps section...
INFO:__main__:Processing section 'PUMPS' from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png...


OCR Row 1: PUMPS .
OCR Row 2: oEEEEEEEEEEEEEEEEEEEEEEEOEOEOEOEOEOECEOECECECECECECE=~=E$7EEEOEOEEEEE—E>XE—¥==eeeE
Number Model Type HHP Efficiency Stroke {in} Liner (in) P-Rating (psi} P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 475 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 475 7500 7100 120 110
At Drilling/Circ Rate Annular Velocity
Drilling/Circ Rate 1 4325 PSl @ 134 SPM 2.63 Gal/Stoke 351.76 GPM 8.38 BPM 468.11 DC 340.61 DP
Drilling/Circ Rate 2 4475 PSl @ 134 SPM 2.63 Gal/Stoke 351.76 GPM 8.38 BPM 468.11 DC 340.61 DP
Pump lines from block: ['PUMPS .', 'oEEEEEEEEEEEEEEEEEEEEEEEOEOEOEOEOEOECEOECECECECECECE=~=E$7EEEOEOEEEEE—E>XE—¥==eeeE', 'Number Model Type HHP Efficiency Stroke {in} Liner (in) P-Rating (psi} P-Limit (psi) SPM Rating SPM Limit', '1 BOMCO TRIPLEX 1600 95 12.000 475 7500 7100 120 110', '3 BOMCO TRIPLEX 1600 95 12.000 475 7500 7100 120 110']
Drilling lines from block: ['At Drilling/Circ Rate Annular Velocity', 'Drilling/Circ Rate 1 4325 PSl @ 1

ERROR:PumpsPipeline.parse_pump_lines:Pump header not detected in OCR text.
INFO:__main__:Section 'PUMPS' processed successfully.
INFO:__main__:Aggregated results saved to /dbfs/mnt/mini-proj-dd/final_results/aggregated_data.json and /dbfs/mnt/mini-proj-dd/final_results/aggregated_data.csv.


Projection segmentation output: ['Neen ———<_ =', 'Number ; Model ; Type ; HHP ; Efficiency ; Stroke (in) ; Liner (in) ; P-Rating (psi) ; P-Limit (psi) ; SPM Rating ; SPM Limit', '+ | _Bomco | __TRIPLEX Of BOO fH B00 S900 HOOT BO', '3 somo | triptex | a600—+[ SiS SSSC*dCSSC~iO—SC~SCSC“‘“IS~S~C*YSCSSCSCSOO—SdSSt00—S*~SSa0 Sit 7 =e 1600 95 12.000 475 oo 7100 120 110', 'palace nate? jas psi asd 3 Gal/toke 376 GPM 338 BeM| ~a68.17 OC 340.61 BP']
Final pump lines after re-segmentation:
Number

Model

Type

HHP

Efficiency

Stroke (in)

Liner (in}

P-Rating (psi)

P-Limit (psi)

SPM Rating

SPM Limit

1

BOMCO

TRIPLEX

1600

95

12.000

4.75

7500

7100

120

110

2

BOMCO

TRIPLEX

1600

95

12.000

4.75

7500

7100

120

110

3

BOMCO

TRIPLEX

1600

95

12.000

4.75

7500

7100

120

110

At Drilling/Cire Rate

Annular Velocity

4325 PSI

134

SPM

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Cire Rate 1

@

2.63 Gal/Stoke

Drilling/Cire Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Sto

In [0]:
import os
import re
import cv2
import json
import logging
import numpy as np
import pytesseract
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# Helper: Convert DBFS paths to local paths
# -----------------------------------------------------------------------------
def dbfs_to_local_path(dbfs_path: str) -> str:
    if dbfs_path.startswith("dbfs:/"):
        return "/dbfs" + dbfs_path.replace("dbfs:", "")
    return dbfs_path

# -----------------------------------------------------------------------------
# (Optional) Helper: Display image (for debugging only)
# -----------------------------------------------------------------------------
def show_image(title: str, img, size=(10,10), cmap=None):
    plt.figure(figsize=size)
    plt.title(title)
    if cmap:
        plt.imshow(img, cmap=cmap)
    else:
        if len(img.shape) == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

# -----------------------------------------------------------------------------
# PumpsPipeline Class: encapsulates all Pumps section processing logic.
# -----------------------------------------------------------------------------
class PumpsPipeline:
    @staticmethod
    def enhance_image(img: np.ndarray) -> np.ndarray:
        """Enhance image for OCR: convert to grayscale, equalize, and apply bilateral filtering."""
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        eq = cv2.equalizeHist(gray)
        filtered = cv2.bilateralFilter(eq, d=9, sigmaColor=75, sigmaSpace=75)
        return filtered

    @staticmethod
    def binarize_image(img: np.ndarray) -> (np.ndarray, np.ndarray):
        """Return both Otsu and adaptive thresholded images."""
        _, otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        adaptive = cv2.adaptiveThreshold(img, 255, 
                                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY, 15, 10)
        return otsu, adaptive

    @staticmethod
    def detect_rows_via_morph_ops(img: np.ndarray, debug: bool = False) -> list:
        """Detect row boundaries via morphological operations."""
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        morph = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=2)
        inv = cv2.bitwise_not(morph)
        contours, _ = cv2.findContours(inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 50 and h < 20:
                boxes.append((x, y, w, h))
        boxes.sort(key=lambda b: b[1])
        # Derive full row boundaries from the y coordinates.
        y_coords = []
        for (x, y, w, h) in boxes:
            y_coords.append(y)
            y_coords.append(y+h)
        y_coords = sorted(list(set(y_coords)))
        h_img, w_img = gray.shape
        if y_coords[0] > 5:
            y_coords.insert(0, 0)
        if abs(y_coords[-1] - h_img) > 5:
            y_coords.append(h_img)
        row_boxes = []
        for i in range(len(y_coords) - 1):
            if y_coords[i+1] - y_coords[i] >= 10:
                row_boxes.append((0, y_coords[i], w_img, y_coords[i+1]-y_coords[i]))
        if debug:
            print(f"Detected {len(row_boxes)} rows via morphology.")
        return row_boxes

    @staticmethod
    def ocr_on_rows(img: np.ndarray, row_boxes: list, debug: bool = False) -> list:
        """Perform OCR on each detected row (using --psm 6) and return the list of text strings."""
        texts = []
        for (x, y, w, h) in row_boxes:
            roi = img[y:y+h, x:x+w]
            gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            pil_roi = Image.fromarray(bw)
            text = pytesseract.image_to_string(pil_roi, config="--psm 6").strip()
            texts.append(text)
        return texts

    @staticmethod
    def multi_psm_ocr(roi: np.ndarray, debug: bool = False) -> str:
        """Run OCR on ROI using multiple PSM modes (6 and 11) and return the best result."""
        enhanced = PumpsPipeline.enhance_image(roi)
        otsu, _ = PumpsPipeline.binarize_image(enhanced)
        pil_img = Image.fromarray(otsu)
        psm_modes = ["6", "11"]
        results = {}
        for mode in psm_modes:
            config = f"--psm {mode}"
            text = pytesseract.image_to_string(pil_img, config=config).strip()
            lines = [l for l in text.splitlines() if l.strip()]
            results[mode] = (text, lines)
        best_mode = max(psm_modes, key=lambda m: sum(1 for l in results[m][1] if l.strip() and l.strip()[0].isdigit()))
        return results[best_mode][0]

    @staticmethod
    def segment_rows_via_projection(roi: np.ndarray, debug: bool = False) -> list:
        """Segment ROI into rows via horizontal projection."""
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        proj = np.sum(binary, axis=1)
        gap_thresh = np.max(proj) * 0.2
        gap_indices = np.where(proj < gap_thresh)[0]
        if len(gap_indices) == 0:
            pil_img = Image.fromarray(binary)
            return [pytesseract.image_to_string(pil_img, config="--psm 7").strip()]
        gap_segments = []
        start = gap_indices[0]
        prev = gap_indices[0]
        for idx in gap_indices[1:]:
            if idx - prev > 1:
                gap_segments.append((start, prev))
                start = idx
            prev = idx
        gap_segments.append((start, prev))
        boundaries = [int((s+e)//2) for s, e in gap_segments]
        boundaries = [0] + boundaries + [binary.shape[0]]
        row_texts = []
        for i in range(len(boundaries)-1):
            r1, r2 = boundaries[i], boundaries[i+1]
            if r2 - r1 < 10:
                continue
            row_img = roi[r1:r2, :]
            pil_row = Image.fromarray(row_img)
            text = pytesseract.image_to_string(pil_row, config="--psm 7").strip()
            if text:
                row_texts.append(text)
        return row_texts

    @staticmethod
    def split_block_to_sections(block_text: str) -> (list, list):
        """
        Split the combined OCR block into pump and drilling sections.
        Lines containing drilling keywords are filtered out of the pump section.
        """
        lines = [l.strip() for l in block_text.splitlines() if l.strip()]
        pump_lines = []
        drilling_lines = []
        section = None
        for line in lines:
            low = line.lower()
            if "drilling/circ" in low:
                section = "drilling"
                drilling_lines.append(line)
                continue
            if section == "drilling":
                drilling_lines.append(line)
            elif section == "pump":
                pump_lines.append(line)
            else:
                if "pumps" in low:
                    section = "pump"
                    continue
                # If the line contains drilling keywords, send it to drilling section.
                if "drilling" in low or "circ" in low:
                    section = "drilling"
                    drilling_lines.append(line)
                elif line and line[0].isdigit():
                    pump_lines.append(line)
        return pump_lines, drilling_lines

    @staticmethod
    def parse_pump_lines(lines: list) -> list:
        """
        Parse pump lines into structured pump rows.
        If lines appear tokenized (few spaces), group tokens in batches of 11.
        Otherwise, split each line on multiple spaces.
        """
        trimmed = [l.strip() for l in lines if l.strip()]
        avg_spaces = np.mean([l.count(" ") for l in trimmed]) if trimmed else 0
        pump_rows = []
        if avg_spaces < 1:
            tokens = trimmed
            if tokens and re.match(r'number', tokens[0].lower()):
                tokens = tokens[11:]
            for i in range(0, len(tokens), 11):
                group = tokens[i:i+11]
                if len(group) < 11:
                    group += [""] * (11 - len(group))
                pump_rows.append({
                    "Number": group[0],
                    "Model": group[1],
                    "Type": group[2],
                    "HHP": group[3],
                    "Efficiency": group[4],
                    "Stroke (in)": group[5],
                    "Liner (in)": group[6],
                    "P-Rating (psi)": group[7],
                    "P-Limit (psi)": group[8],
                    "SPM Rating": group[9],
                    "SPM Limit": group[10]
                })
        else:
            header_found = False
            for line in trimmed:
                low = line.lower()
                if "number" in low and "model" in low:
                    header_found = True
                    continue
                if header_found or (line and line[0].isdigit()):
                    tokens = re.split(r'\s{2,}|\|', line)
                    if len(tokens) < 2:
                        tokens = line.split()
                    if len(tokens) < 11:
                        tokens += [""] * (11 - len(tokens))
                    pump_rows.append({
                        "Number": tokens[0],
                        "Model": tokens[1],
                        "Type": tokens[2],
                        "HHP": tokens[3],
                        "Efficiency": tokens[4],
                        "Stroke (in)": tokens[5],
                        "Liner (in)": tokens[6],
                        "P-Rating (psi)": tokens[7],
                        "P-Limit (psi)": tokens[8],
                        "SPM Rating": tokens[9],
                        "SPM Limit": tokens[10]
                    })
        return pump_rows

    @staticmethod
    def parse_drilling_lines(lines: list) -> list:
        """Parse drilling/circ rate lines into structured rows."""
        drilling_rows = []
        for line in lines:
            tokens = line.split()
            if len(tokens) < 8:
                continue
            rate_id = tokens[2] if tokens[2].isdigit() else ""
            try:
                pressure = tokens[3] if len(tokens) > 3 else ""
                spm = tokens[6] if len(tokens) > 6 else ""
                gal_stroke = tokens[8] if len(tokens) > 8 else ""
                gpm = tokens[10] if len(tokens) > 10 else ""
                bpm = tokens[12] if len(tokens) > 12 else ""
                dc = tokens[14] if len(tokens) > 14 else ""
                dp = tokens[16] if len(tokens) > 16 else ""
            except Exception:
                pressure, spm, gal_stroke, gpm, bpm, dc, dp = ("",)*7
            drilling_rows.append({
                "RateID": rate_id,
                "Pressure": pressure,
                "SPM": spm,
                "Gal_Stroke": gal_stroke,
                "GPM": gpm,
                "BPM": bpm,
                "DC": dc,
                "DP": dp,
                "Raw": line
            })
        return drilling_rows

    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        """
        Main processing method.
        Reads image, applies OCR using enhanced techniques, reassembles pump rows, and
        returns a tuple (data_json, df) with structured data and a DataFrame.
        """
        local_path = dbfs_to_local_path(image_path)
        img = cv2.imread(local_path)
        if img is None:
            raise FileNotFoundError(f"Unable to read image at: {image_path}")
        
        row_boxes = PumpsPipeline.detect_rows_via_morph_ops(img, debug=debug)
        row_texts = PumpsPipeline.ocr_on_rows(img, row_boxes, debug=debug)
        block_text = "\n".join(row_texts)
        
        pump_lines, drilling_lines = PumpsPipeline.split_block_to_sections(block_text)
        
        # Re-segment pump area if a dedicated ROI is found.
        pump_roi = None
        for i, text in enumerate(row_texts):
            if "number" in text.lower() and "model" in text.lower():
                bx = row_boxes[i]
                pump_roi = img[bx[1]:bx[1]+bx[3], bx[0]:bx[0]+bx[2]]
                break
        if pump_roi is not None:
            pump_block = PumpsPipeline.multi_psm_ocr(pump_roi, debug=debug)
            resegmented = PumpsPipeline.segment_rows_via_projection(pump_roi, debug=debug)
            if len(resegmented) > len(pump_block.splitlines()):
                final_pump_lines = resegmented
            else:
                final_pump_lines = pump_block.splitlines()
            pump_lines = final_pump_lines
        
        pumps = PumpsPipeline.parse_pump_lines(pump_lines)
        drilling = PumpsPipeline.parse_drilling_lines(drilling_lines)
        data_json = {"PUMPS": pumps, "DrillingCircRates": drilling}
        df = pd.DataFrame(pumps)
        return data_json, df

# -----------------------------------------------------------------------------
# PipelineRunner Class: Integrates PumpsPipeline into the main process.
# -----------------------------------------------------------------------------
class PipelineRunner:
    def __init__(self, image_path: str, debug: bool = False):
        self.image_path = image_path
        self.debug = debug
        self.logger = logging.getLogger("PipelineRunner")
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
            self.logger.addHandler(handler)
    
    def run(self):
        try:
            data_json, df = PumpsPipeline.process(self.image_path, debug=self.debug)
            print("----- Pumps Extraction JSON Output -----")
            print(json.dumps(data_json, indent=4))
            df.to_csv("pumps_extracted.csv", index=False)
            self.logger.info("pumps_extracted.csv saved.")
            # Save drilling data to CSV.
            df_drill = pd.DataFrame(data_json["DrillingCircRates"])
            df_drill.to_csv("drilling_rates_extracted.csv", index=False)
            self.logger.info("drilling_rates_extracted.csv saved.")
        except Exception as e:
            self.logger.exception("Error during pipeline processing.")

# -----------------------------------------------------------------------------
# Final Integrated Main Pipeline
# -----------------------------------------------------------------------------
def main(debug: bool = False):
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    logger.info("Starting main pipeline execution for Pumps section...")
    
    image_paths = {
        "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png"
    }
    
    pipelines = {
        "PUMPS": PumpsPipeline.process
    }
    
    output_folder = dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
    os.makedirs(output_folder, exist_ok=True)
    aggregated_json = {}
    aggregated_df = pd.DataFrame()
    
    for section, img_path in image_paths.items():
        func = pipelines.get(section)
        if func is None:
            logger.info(f"Skipping section '{section}' — pipeline not implemented.")
            continue
        try:
            logger.info(f"Processing section '{section}' from {img_path}...")
            data_json, df = func(img_path, debug=debug)
            safe_section = section.replace(" ", "_").lower()
            json_file = os.path.join(output_folder, f"{safe_section}.json")
            with open(json_file, "w") as f:
                json.dump(data_json, f, indent=4)
            if df is not None and not df.empty:
                aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
                csv_file = os.path.join(output_folder, f"{safe_section}.csv")
                df.to_csv(csv_file, index=False)
            aggregated_json[section] = data_json
            logger.info(f"Section '{section}' processed successfully.")
        except Exception as e:
            logger.exception(f"Error processing section '{section}': {e}")
    
    agg_json_path = os.path.join(output_folder, "aggregated_data.json")
    with open(agg_json_path, "w") as f:
        json.dump(aggregated_json, f, indent=4)
    agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
    aggregated_df.to_csv(agg_csv_path, index=False)
    logger.info(f"Aggregated results saved to {agg_json_path} and {agg_csv_path}.")
    print("----- Aggregated JSON Output -----")
    print(json.dumps(aggregated_json, indent=4))

# -----------------------------------------------------------------------------
# Entry Point
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    main(debug=False)


----- Aggregated JSON Output -----
{
    "PUMPS": {
        "PUMPS": [
            {
                "Number": "1",
                "Model": "BOMCO",
                "Type": "TRIPLEX",
                "HHP": "1600",
                "Efficiency": "95",
                "Stroke (in)": "12.000",
                "Liner (in)": "4.75",
                "P-Rating (psi)": "7500",
                "P-Limit (psi)": "7100",
                "SPM Rating": "120",
                "SPM Limit": "110"
            },
            {
                "Number": "2",
                "Model": "BOMCO",
                "Type": "TRIPLEX",
                "HHP": "1600",
                "Efficiency": "95",
                "Stroke (in)": "12.000",
                "Liner (in)": "4.75",
                "P-Rating (psi)": "7500",
                "P-Limit (psi)": "7100",
                "SPM Rating": "120",
                "SPM Limit": "110"
            },
            {
                "Number": "3",
                "Model":

In [0]:
import os
import re
import pytesseract
import logging
import json
import pandas as pd
from PIL import Image

# -----------------------------------------------------------------------------
# 1) Minimal Logger Configuration
# -----------------------------------------------------------------------------
logger = logging.getLogger("PumpExtractor")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    logger.addHandler(handler)

# -----------------------------------------------------------------------------
# 2) Read Image
# -----------------------------------------------------------------------------
def read_image(image_path):
    """
    Reads the image from a local or DBFS path and returns a PIL Image.
    """
    if image_path.startswith("dbfs:/"):
        # Convert "dbfs:/mnt/..." to "/dbfs/mnt/..."
        local_path = "/dbfs" + image_path.replace("dbfs:", "")
    else:
        local_path = image_path

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    
    img = Image.open(local_path)
    logger.info(f"Image loaded from {local_path} with size {img.size}")
    return img

# -----------------------------------------------------------------------------
# 3) Perform OCR
# -----------------------------------------------------------------------------
def perform_ocr(img):
    """
    Performs OCR on the given PIL image and returns the raw text.
    """
    text = pytesseract.image_to_string(img)
    logger.info("OCR extraction complete.")
    return text

# -----------------------------------------------------------------------------
# 4) Parse Pumps Table
# -----------------------------------------------------------------------------
def parse_pumps_table(ocr_text):
    """
    Parses the pumps table from the OCR text.
    Expected pump rows look like:
      Number Model Type   HHP  Efficiency  Stroke(in)  Liner(in)  P-Rating(psi)  P-Limit(psi)  SPM Rating  SPM Limit
      1      BOMCO TRIPLEX 1600 95       12.000       475        7500           7100          120         110
      2      BOMCO TRIPLEX 1600 95       12.000       475        7500           7100          120         110
      3      BOMCO TRIPLEX 1600 95       12.000       475        7500           7100          120         110
    """
    pump_pattern = re.compile(
        r"^(\d+)\s+(BOMCO)\s+(TRIPLEX)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
        re.IGNORECASE
    )
    lines = ocr_text.splitlines()
    pumps = []
    for line in lines:
        line = line.strip()
        match = pump_pattern.match(line)
        if match:
            number, model, pump_type, hhp, efficiency, stroke, liner, p_rating, p_limit, spm_rating, spm_limit = match.groups()
            pumps.append({
                "Number": number,
                "Model": model.upper(),
                "Type": pump_type.upper(),
                "HHP": hhp,
                "Efficiency": efficiency,
                "Stroke (in)": stroke,
                "Liner (in)": liner,
                "P-Rating (psi)": p_rating,
                "P-Limit (psi)": p_limit,
                "SPM Rating": spm_rating,
                "SPM Limit": spm_limit
            })
    return pumps

# -----------------------------------------------------------------------------
# 5) Parse Drilling/Circ Rates
# -----------------------------------------------------------------------------
def parse_drilling_circ_rates(ocr_text):
    """
    Parses drilling/circ rate lines from the OCR text.
    Expected format (example):
      Drilling/Circ Rate 1 4325 PSI @ 134 SPM 2.63 Gal/Stoke 351.76 GPM 8.38 BPM 468.11 DC 340.61 DP
      Drilling/Circ Rate 2 4475 PSI @ 134 SPM 2.63 Gal/Stoke 351.76 GPM 8.38 BPM 468.11 DC 340.61 DP
    """
    circ_pattern = re.compile(
        r"Drilling/Circ\s+Rate\s+(\d+)\s+(\d+)\s+PSI\s+@\s+(\d+)\s+SPM\s+([\d.]+)\s+Gal/Stoke\s+([\d.]+)\s+GPM\s+([\d.]+)\s+BPM\s+([\d.]+)\s+DC\s+([\d.]+)\s+DP",
        re.IGNORECASE
    )
    lines = ocr_text.splitlines()
    circ_rates = []
    for line in lines:
        line = line.strip()
        match = circ_pattern.match(line)
        if match:
            rate_id, pressure, spm, gal_stroke, gpm, bpm, dc, dp = match.groups()
            circ_rates.append({
                "RateID": rate_id,
                "Pressure": pressure,
                "SPM": spm,
                "Gal_Stroke": gal_stroke,
                "GPM": gpm,
                "BPM": bpm,
                "DC": dc,
                "DP": dp
            })
    return circ_rates

# -----------------------------------------------------------------------------
# 6) Main Pipeline Function
# -----------------------------------------------------------------------------
def main_pipeline():
    # Adjust this path as needed
    image_path = "/dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png"
    try:
        img = read_image(image_path)
    except FileNotFoundError as e:
        logger.error(e)
        return

    # Perform OCR on the entire image
    ocr_text = perform_ocr(img)
    logger.info(f"OCR Text:\n{ocr_text}\n")
    print("----- Full OCR Extracted Text -----")
    print(ocr_text)

    # Parse the pumps table and drilling/circ rates from the OCR text
    pumps = parse_pumps_table(ocr_text)
    circ_rates = parse_drilling_circ_rates(ocr_text)

    # Print extracted pumps and drilling rates (for debugging)
    print("----- Extracted Pumps Table -----")
    print(json.dumps(pumps, indent=4))
    print("----- Extracted Drilling/Circ Rates -----")
    print(json.dumps(circ_rates, indent=4))

    final_data = {
        "PUMPS": pumps,
        "DrillingCircRates": circ_rates
    }

    # Convert to DataFrames for saving as CSV (if needed)
    df_pumps = pd.DataFrame(pumps)
    df_circ = pd.DataFrame(circ_rates)

    logger.info("=== Pumps DataFrame ===")
    print(df_pumps)
    logger.info("=== Drilling/Circ Rates DataFrame ===")
    print(df_circ)

    logger.info("=== Final JSON ===")
    final_json = json.dumps(final_data, indent=4)
    print(final_json)

    # Save results
    output_folder = "/dbfs/mnt/mini-proj-dd/final_results"
    os.makedirs(output_folder, exist_ok=True)
    csv_pumps_path = os.path.join(output_folder, "pumps.csv")
    csv_circ_path = os.path.join(output_folder, "drilling_circ_rates.csv")
    json_path = os.path.join(output_folder, "pumps_drilling_circ.json")
    df_pumps.to_csv(csv_pumps_path, index=False)
    df_circ.to_csv(csv_circ_path, index=False)
    with open(json_path, "w") as f:
        json.dump(final_data, f, indent=4)
    logger.info(f"Pumps CSV saved to: {csv_pumps_path}")
    logger.info(f"Drilling/Circ Rates CSV saved to: {csv_circ_path}")
    logger.info(f"JSON saved to: {json_path}")

# -----------------------------------------------------------------------------
# Entry Point
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    main_pipeline()


INFO: Image loaded from /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png with size (2502, 276)
INFO:PumpExtractor:Image loaded from /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png with size (2502, 276)
INFO: OCR extraction complete.
INFO:PumpExtractor:OCR extraction complete.
INFO: OCR Text:
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP


INFO:PumpExtractor:OCR Text:
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 710

----- Full OCR Extracted Text -----
Number Type Efficiency Stroke (in) Liner (in) P-Rating (psi) P-Limit (psi) SPM Rating SPM Limit
1 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
2 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110
3 BOMCO TRIPLEX 1600 95 12.000 4.75 7500 7100 120 110

At Drilling/Circ Rate

Annular Velocity

Drilling/Circ Rate 1

4325 PS!

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

Drilling/Circ Rate 2

4475 PSI

@

134

SPM

2.63 Gal/Stoke

351.76 GPM

8.38 BPM

468.11 DC

340.61 DP

----- Extracted Pumps Table -----
[
    {
        "Number": "1",
        "Model": "BOMCO",
        "Type": "TRIPLEX",
        "HHP": "1600",
        "Efficiency": "95",
        "Stroke (in)": "12.000",
        "Liner (in)": "4.75",
        "P-Rating (psi)": "7500",
        "P-Limit (psi)": "7100",
        "SPM Rating": "120",
        "SPM Limit": "110"
    },
    {
        "Number": "2",
        "Model": "BOMCO",
        "Type": "TRIPLEX",
        "H